# ATUA 2026# — Spatial Regression and Geographically Weighted Regression
## Author: Chrysa Lamprinopoulou

## Introduction

**Data:** (1) Inside Airbnb listings for Edinburgh, UK —  (2) DataZone Boundaries 2022 for Scotland — [data.gov.uk](https://www.data.gov.uk/dataset/afe1aca6-bea2-4283-8847-eecf98ab41a4/data-zone-boundaries-2022).

**Dependent variable (Y):** Log-transformed listing price. **Research question:** What factors determine Airbnb pricing, and which spatial model best captures the underlying geographic dependence? The analysis proceeds from OLS through spatial econometric models (SEM, SAR, SDM) to local regression (GWR, MGWR).

This assignment investigates the determinants of Airbnb prices in Edinburgh using rich listing-level data from Inside Airbnb [insideairbnb.com](https://insideairbnb.com/get-the-data/) and small-area neighbourhood statistics based on Scotland’s 2022 Data Zone boundaries [data.gov.uk](https://www.data.gov.uk/dataset/afe1aca6-bea2-4283-8847-eecf98ab41a4/data-zone-boundaries-2022). The dependent variable is the log of the advertised price, allowing proportional effects of various predictors to be examined. 

The **core research questions** are: 
(1) which listing, host, and neighbourhood characteristics systematically influence Airbnb prices in Edinburgh and; 
(2) to what extent is pricing shaped by spatial dependence, that is, by interactions between nearby listings and shared local context.

To answer these questions, the analysis begins with a global OLS benchmark and then compares alternative spatial econometric models (SEM, SAR, SDM) before moving to local models (GWR, MGWR) that explicitly allow price–covariate relationships to vary across Edinburgh’s urban space

## Literature review

To provide an accurate analysis of Edinburgh's Airbnb market, research suggests that the determinants of listing prices can be grouped into five broad categories, including nightly listing price, property features, listing/host characteristics, tourism-related context, and spatial location (Wang & Nicolau, 2017). However, empirical findings often show that Airbnb prices primarily depend on three core aspects: physical attributes (i.e., what the property offers), perceptual factors (i.e., why users prefer it), and location (i.e., where it sits) (Perez-Sanchez et al., 2018). In the specific context of Edinburgh, physical attributes—particularly the number of bedrooms and total accommodation capacity—serve as the strongest predictors of the nightly rate (Voltes-Dorta & Sánchez-Medina, 2020). Furthermore, the city's unique geography creates significant spatial heterogeneity, where location-specific variables like proximity to major tourist landmarks in the Old Town or transport links in the West End command substantial price premiums (Luo et al., 2023). Finally, perceptual factors such as host reputation and "Superhost" status function as critical quality signals, allowing experienced hosts in mature markets like Edinburgh to leverage positive reviews to maintain higher pricing levels compared to newer listings (Sainaghi, 2021).

In [3]:
#pip install nbformat

In [4]:
import nbformat

nb_path = "/Users/xrysa/Desktop/MSc_in_Urban_Analytics/Advanced_Topics/ATUA2026_UV/3045603L_ATUA2026.ipynb"

nb = nbformat.read(nb_path, as_version=4)

word_markdown = 0
word_heading = 0
word_code = 0

for cell in nb.cells:
    src = cell.get('source', '')

    # normalise source to a single string
    if isinstance(src, list):
        src = ''.join(src)

    if cell['cell_type'] == 'markdown':
        for line in src.splitlines():
            stripped = line.lstrip()
            if stripped.startswith('#'):
                tokens = stripped.lstrip('#').strip().split()
                word_heading += len(tokens)
            else:
                tokens = stripped.split()
                word_markdown += len(tokens)

    elif cell['cell_type'] == 'code':
        for line in src.splitlines():
            # remove comments before counting
            code = line.split('#', 1)[0]
            tokens = code.strip().split()
            word_code += len(tokens)

print(f"{word_markdown} Words in notebooks' markdown")
print(f"{word_heading} Words in notebooks' heading")
print(f"{word_code} Words in notebooks' code")


ModuleNotFoundError: No module named 'nbformat'

In [ ]:
#pip install libpysal

In [ ]:
#pip install esda

In [ ]:
pip install spreg

In [ ]:
#pip install splot


In [ ]:

# Python `uv` environment used throughout this assignemnt. 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import geopandas as gpd
import seaborn as sns
from libpysal.weights import DistanceBand, KNN
from esda.moran import Moran
from spreg import OLS, ML_Error, ML_Lag, GM_Error, GM_Lag
from scipy import stats
from shapely.geometry import Point
from splot.esda import moran_scatterplot, plot_moran
import contextily as ctx
from matplotlib.ticker import FuncFormatter
import pyproj
from libpysal.weights import lag_spatial
from scipy.stats import norm

### 1. Load and Explore the Dataset

In [ ]:
#df = pd.read_csv(r'Spatial_Regression/Data/listings.csv.gz')
df = pd.read_csv('/Users/xrysa/Desktop/MSc_in_Urban_Analytics/Advanced_Topics/2_3_week_excersise/listings.csv.gz')
df.to_csv("/Users/xrysa/Desktop/MSc_in_Urban_Analytics/Advanced_Topics/2_3_week_excersise/unzipped_listings.csv", index=False)
df.shape
df.head(2)
#print(df.columns.tolist())

In [ ]:
df.shape


### 2. Data Cleaning

In [ ]:
# Clean the dolar symbol from price 
#df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)

In [ ]:

# Select all variables
cols = ['id', 'host_is_superhost', 
'host_listings_count', 'bathrooms_text',
'host_total_listings_count', 'host_identity_verified',
'latitude', 'longitude', 'instant_bookable',
'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
'price', 'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d', 
'review_scores_rating',  'review_scores_cleanliness',
'review_scores_communication', 'review_scores_location',  
'calculated_host_listings_count', 'reviews_per_month',  
'host_response_rate','host_acceptance_rate'
]

df_clean= df[cols].copy()


In [ ]:
# Take a quick look at summary statistics for price before cleaning 
df_clean['price'].describe()

In [ ]:
df_clean.shape

Dollar and percentage symbols were stripped from `price`, `host_response_rate`, and `host_acceptance_rate`; `bathrooms` was extracted numerically from `bathrooms_text`; binary variables (`is_superhost`, `is_instant_bookable`, `has_identity_verified`) were encoded as 0/1; and room-type dummies were created with "Entire home/apt" as the reference category.

In [ ]:
# Clean the dolar symbol from price 
df_clean['price'] = df_clean['price'].replace(r'[\$,]', '', regex=True).astype(float)
df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)

# Clean the % symbol from host_response_rate and host_acceptance_rate
df_clean['host_response_rate'] = df_clean['host_response_rate'].replace(r'[\%,]', '', regex=True).astype(float)
df_clean['host_acceptance_rate'] = df_clean['host_acceptance_rate'].replace(r'[\%,]', '', regex=True).astype(float)
df['host_response_rate'] = df_clean['host_response_rate'].replace(r'[\%,]', '', regex=True).astype(float)
df['host_acceptance_rate'] = df_clean['host_acceptance_rate'].replace(r'[\%,]', '', regex=True).astype(float)

# Clean bathrooms 
df_clean['bathrooms'] = df_clean['bathrooms_text'].str.extract(r'(\d+\.?\d*)').astype(float)
df_clean = df_clean.drop(columns='bathrooms_text')

# Encode binary variables 
df_clean['is_superhost'] = (df_clean['host_is_superhost'] == 't').astype(int)
df_clean['is_instant_bookable'] = (df_clean['instant_bookable'] == 't').astype(int)
df_clean['has_identity_verified'] = (df_clean['host_identity_verified'] == 't').astype(int)
df_clean = df_clean.drop(columns=['host_is_superhost', 'instant_bookable', 'host_identity_verified'])

# Create room type dummies 
# Reference category: 'Entire home/apt'
room_dummies = pd.get_dummies(df_clean['room_type'], prefix='room', drop_first=False)

# Keep interpretable dummies (drop 'Entire home/apt' as reference)
if 'room_Entire home/apt' in room_dummies.columns:
    room_dummies = room_dummies.drop(columns='room_Entire home/apt')
df_clean = pd.concat([df_clean, room_dummies], axis=1)

In [ ]:
df_clean[['price', 'host_response_rate', 'host_acceptance_rate', 'bathrooms']].describe()

In [ ]:
df[['price', 'host_response_rate', 'host_acceptance_rate', 'bathrooms']].describe()

In [ ]:
df_clean.shape

In [ ]:
# Detect and drop missing values in key variables

key_cols = ['id', 'is_superhost', 
'host_listings_count', 
'host_total_listings_count', 'has_identity_verified',
'latitude', 'longitude', 'is_instant_bookable',
'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
'price', 'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d', 
'review_scores_rating',  'review_scores_cleanliness',
'review_scores_communication', 'review_scores_location',  
'calculated_host_listings_count', 'reviews_per_month',  
'host_response_rate','host_acceptance_rate'
]


# Detect missing values in key variables
n_rows = len(df_clean)
print (n_rows)
missing_summary = pd.DataFrame({
    'Missing count': df_clean[key_cols].isna().sum(),
    'Missing percentage (%)': ((df_clean[key_cols].isna().sum() / len(df_clean) * 100).round(1))
})
print(missing_summary)

#Drop missing values in key variables
df_clean = df_clean.dropna(subset=key_cols)



In [ ]:
df_clean.shape

In [ ]:
# Explore price outliers
fig, ax = plt.subplots(figsize=(8, 5))

ax.boxplot(df_clean['price'], vert=True)
ax.set_title('Price Distribution', fontsize=13)
ax.set_ylabel('Price (£)')

plt.tight_layout()
plt.show()

In [ ]:
df_clean.shape

The boxplot confirms right-skewed outliers. The top 1% of prices is removed to prevent extreme values from distorting the regression.

In [ ]:
# Remove top 1% price outliers
price_99th = df_clean['price'].quantile(0.99)
n_before = len(df_clean)

df_clean = df_clean[df_clean['price'] <= price_99th]
df_clean = df_clean.reset_index(drop=True)

print(f"99th percentile price: £{price_99th:.2f}")
print(f"After removing top 1%: {len(df_clean)} listings")
print(f"Listings removed: {n_before - len(df_clean)}")

In [ ]:
df_clean.shape

The log transformation reduces skewness and stabilises variance, yielding a distribution better suited to OLS assumptions.

In [ ]:
# Log-transform price to reduce skewness and stabilise variance for regression analysis as demonstated below
df_clean['log_price'] = np.log(df_clean['price'])

In [ ]:
# Combine raw and cleaned price into one table
price_compare = pd.concat(
    [
        df['price'].rename('price_raw'),  # before cleaning 
        df_clean['price'].rename('price_clean'), # after cleaning 
        df_clean['log_price'].rename('log_price')  # optional
    ],
    axis=1
)

# Summary stats for all three
price_compare.describe().T.round(2) # T displays variables as rows which is nicer to read



In [ ]:
df_clean.shape

In [ ]:
# Visualising the distribution of price before and after log transformation.

price = df_clean['price']
log_price = df_clean['log_price']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot A: Original Price (after cleaning)
sns.histplot(price, bins=50, kde=False, color='skyblue', stat='density', ax=axes[0], label='Data')
# Overlay Normal Curve (based on original mean/std)
mu, std = norm.fit(price)
xmin, xmax = axes[0].get_xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, std)
axes[0].plot(x, p, 'r', linewidth=2, label='Normal Dist.')

axes[0].set_title(f'Original Clean Price Distribution\n(Skewness: {price.skew():.2f})', fontsize=14)
axes[0].set_xlabel('Price ($)')
axes[0].legend()

# Plot B: Log Transformed Price 
sns.histplot(log_price, bins=50, kde=False, color='green', stat='density', ax=axes[1], label='Data')
# Overlay Normal Curve (based on log mean/std)
mu_log, std_log = norm.fit(log_price)
xmin, xmax = axes[1].get_xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu_log, std_log)
axes[1].plot(x, p, 'r', linewidth=2, label='Normal Dist.')

axes[1].set_title(f'Log-Transformed Price Distribution\n(Skewness: {log_price.skew():.2f})', fontsize=14)
axes[1].set_xlabel('Log Price')
axes[1].legend()


plt.tight_layout()
plt.show()

After log transformation the distribution is approximately symmetric, suitable for regression analysis.

In [ ]:
geometry = [Point(xy) for xy in zip(df_clean['longitude'], df_clean['latitude'])]
gdf = gpd.GeoDataFrame(df_clean, geometry=geometry, crs='EPSG:4326')

gdf_web = gdf.to_crs('EPSG:3857')

fig, ax = plt.subplots(figsize=(10, 8))

gdf_web.plot(
    ax=ax, column='log_price', cmap='YlOrRd',
    markersize=5, alpha=0.6, legend=True,
    legend_kwds={'label': 'Log Price (£)', 'shrink': 0.7}
)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_title(f'Spatial Distribution of Log-Transformed Airbnb Listing Prices in Edinburgh', fontsize=14)
ax.set_xlabel('Lon')
ax.set_ylabel('Lat')

transformer = pyproj.Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True)

def lon_formatter(x, pos):
    lon, _ = transformer.transform(x, 0)
    return f'{lon:.2f}°'

def lat_formatter(y, pos):
    _, lat = transformer.transform(0, y)
    return f'{lat:.2f}°'

ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
ax.yaxis.set_major_formatter(FuncFormatter(lat_formatter))

plt.tight_layout()
plt.show()

### 3. Spatial Weight Matrix ($W$)

A KNN ($k = 5$) spatial weight matrix defines neighbourhood structure. KNN is preferred over distance-band weights for point data because it guarantees every listing has exactly $k$ neighbours, avoiding the island problem that can arise with isolated observations.

In [ ]:
# Extract projected coordinates for distance calculations
gdf_proj = gdf.to_crs('EPSG:27700')
coords = np.column_stack((gdf_proj.geometry.x, gdf_proj.geometry.y))

# KNN Weight Matrix with k=5 neighbourhood
w_knn = KNN.from_array(coords, k=5)
w_knn.transform = 'R'  # Row-standardise

print("=== KNN Spatial Weights (k=5) ===")
print(f"Number of observations: {w_knn.n}")
print(f"Mean number of neighbours: {w_knn.mean_neighbors:.1f}")
print(f"Min neighbours: {w_knn.min_neighbors}")
print(f"Max neighbours: {w_knn.max_neighbors}")

### 4. OLS Estimation Before Testing For Spatial Autocorrelation.

In [ ]:
# Dependent variable
y_col = 'log_price'
y = df_clean[[y_col]].values

# Independent variables
room_dummy_cols = [c for c in df_clean.columns if c.startswith('room_') and c != 'room_type']

x_cols = ['is_superhost', 
'host_listings_count', 
'host_total_listings_count', 'has_identity_verified',
'is_instant_bookable',
'accommodates', 'bathrooms', 'bedrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d', 
'review_scores_rating',  'review_scores_cleanliness',
'review_scores_communication', 'review_scores_location',  
'calculated_host_listings_count', 'reviews_per_month',  
'host_response_rate','host_acceptance_rate'
    
] + room_dummy_cols


# Convert room type dummies to int
for col in room_dummy_cols:
    df_clean[col] = df_clean[col].astype(int)

x = df_clean[x_cols].values


### 5. Variance Inflation Factor (VIF)

VIF quantifies multicollinearity; values above 10 indicate problematic inflation. An iterative removal procedure is applied: the highest-VIF variable is dropped, VIFs are recalculated, and the process repeats until all values fall below 10, ideally below 5.

In [ ]:
# VIF Check for Multicollinearity

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Create a dataframe of just the independent variables (X)
X_vif = df_clean[x_cols] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

`review_scores_rating` (VIF ≈ 1301) is removed first, due to its near-perfectly collinearity with some other variables.

In [ ]:
# Remove  review_scores_rating due to high VIF
x_cols_update = ['is_superhost', 
'host_listings_count', 
'host_total_listings_count', 'has_identity_verified',
'is_instant_bookable',
'accommodates', 'bathrooms', 'bedrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d', 
'review_scores_cleanliness',
'review_scores_communication', 'review_scores_location',  
'calculated_host_listings_count', 'reviews_per_month',  
'host_response_rate','host_acceptance_rate'
    
] + room_dummy_cols

# Create an updated dataframe of just the independent variables (X)
X_vif = df_clean[x_cols_update] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

`review_scores_communication` is removed next due to highest remaining VIF.

In [ ]:
# Remove  review_scores_communication accomodates due to high VIF
x_cols_update = ['is_superhost', 
'host_listings_count', 
'host_total_listings_count', 'has_identity_verified',
'is_instant_bookable',
'accommodates','bathrooms', 'bedrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d', 
'review_scores_cleanliness',
'review_scores_location',  
'calculated_host_listings_count', 'reviews_per_month',  
'host_response_rate','host_acceptance_rate'
    
] + room_dummy_cols

# Create an updated dataframe of just the independent variables (X)
X_vif = df_clean[x_cols_update] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

`review_scores_location` is removed because it is collinear with remaining review sub-scores.

In [ ]:
# Remove review_scores_location  due to high VIF
x_cols_update = ['is_superhost', 
'host_listings_count', 
'host_total_listings_count', 'has_identity_verified',
'is_instant_bookable',
'accommodates','bathrooms', 'bedrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d', 
'review_scores_cleanliness',  
'calculated_host_listings_count', 'reviews_per_month',  
'host_response_rate','host_acceptance_rate'
    
] + room_dummy_cols

# Create an updated dataframe of just the independent variables (X)
X_vif = df_clean[x_cols_update] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

`host_response_rate` is removed as related to superhost status.

In [ ]:
# Remove host_response_rate  due to high VIF
x_cols_update = ['is_superhost', 
'host_listings_count', 
'host_total_listings_count', 'has_identity_verified',
'is_instant_bookable',
'accommodates','bathrooms', 'bedrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d', 
'review_scores_cleanliness',  
'calculated_host_listings_count', 'reviews_per_month',  
'host_acceptance_rate'
    
] + room_dummy_cols

# Create an updated dataframe of just the independent variables (X)
X_vif = df_clean[x_cols_update] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

`review_scores_cleanliness` is removed due to collinearity with remaining quality indicators.

In [ ]:
# Remove review_scores_cleanliness due to high VIF
x_cols_update = ['is_superhost', 
'host_listings_count', 
'host_total_listings_count', 'has_identity_verified',
'is_instant_bookable',
'accommodates','bathrooms', 'bedrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d',  
'calculated_host_listings_count', 'reviews_per_month',  
'host_acceptance_rate'
    
] + room_dummy_cols

# Create an updated dataframe of just the independent variables (X)
X_vif = df_clean[x_cols_update] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

`accommodates` is removed as collinear with `bedrooms` and `beds`.

In [ ]:
# Remove accommodates due to high VIF
x_cols_update = ['is_superhost', 
'host_listings_count', 
'host_total_listings_count', 'has_identity_verified',
'is_instant_bookable',
'bathrooms', 'bedrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d',  
'calculated_host_listings_count', 'reviews_per_month',  
'host_acceptance_rate'
    
] + room_dummy_cols

# Create an updated dataframe of just the independent variables (X)
X_vif = df_clean[x_cols_update] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

`host_acceptance_rate` is removed as correlated with host professionalism variables.

In [ ]:
# Remove host_acceptance_rate  due to high VIF
x_cols_update = ['is_superhost', 
'host_listings_count', 
'host_total_listings_count', 'has_identity_verified',
'is_instant_bookable',
'bathrooms', 'bedrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d',  
'calculated_host_listings_count', 'reviews_per_month',     
] + room_dummy_cols

# Create an updated dataframe of just the independent variables (X)
X_vif = df_clean[x_cols_update] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

`bedrooms` is removed as collinear with `beds` and `bathrooms`.

In [ ]:
# Remove bedrooms  due to high VIF
x_cols_update = ['is_superhost', 
'host_listings_count', 
'host_total_listings_count', 'has_identity_verified',
'is_instant_bookable',
'bathrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d',  
'calculated_host_listings_count', 'reviews_per_month',     
] + room_dummy_cols

# Create an updated dataframe of just the independent variables (X)
X_vif = df_clean[x_cols_update] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

`has_identity_verified` is removed to bring VIFs closer to 5.

In [ ]:
# Remove bedrooms  due to high VIF
x_cols_update = ['is_superhost', 
'host_listings_count', 
'host_total_listings_count',
'is_instant_bookable',
'bathrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d',  
'calculated_host_listings_count', 'reviews_per_month',     
] + room_dummy_cols

# Create an updated dataframe of just the independent variables (X)
X_vif = df_clean[x_cols_update] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

`host_listings_count` and `host_total_listings_count` are removed; `calculated_host_listings_count` is retained as the sole host-portfolio measure, because it had no missing values.

In [ ]:
# Remove both  'host_listings_count' and  'host_total_listings_count' due to high VIF
x_cols_update = ['is_superhost', 
'is_instant_bookable',
'bathrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d',  
'calculated_host_listings_count', 'reviews_per_month',     
] + room_dummy_cols

# Create an updated dataframe of just the independent variables (X)
X_vif = df_clean[x_cols_update] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("Variance Inflation Factor (VIF) Check:")
print("-" * 40)
print(vif_data.sort_values('VIF', ascending=False))
print("-" * 40)
print("Note: VIF > 10 indicates high multicollinearity that may distort spatial coefficients.")

The iterative VIF elimination procedure has successfully reduced all remaining VIFS below 10.  This indicates that multicollinearity no longer poses a substantive threat to the stability or interpretability of the regression coefficients. The final set of independent variables retains theoretical coherence across five key dimensions of Airbnb pricing, which seem in full agreement with the findings of the academic literature: host characteristics (`is_superhost`,`calculated_host_listings_count`), property attributes (`bathrooms`, `beds`), booking policies (`minimum_nights`, `maximum_nights`, `availability_365`), demand indicators (`number_of_reviews`, `estimated_revenue_l365d`,  `reviews_per_month`), and room type dummies. This specification now serves as the basis for OLS estimation and subsequent spatial diagnostic testing.

### 6. Run OLS

In [ ]:
# Update X with the new set of independent variables
x_cols = x_cols_update
x = df_clean[x_cols].values

In [ ]:
# Run OLS
ols = OLS(
    y, x,
    w=w_knn,
    name_y=y_col,
    name_x=x_cols,
    name_ds='Edinburgh Airbnb',
    spat_diag=True  # Request spatial diagnostics including LM tests
)

print(ols.summary)

### 7. Spatial Autocorrelation in OLS Residuals

Global Moran's I tests whether OLS residuals are spatially clustered rather than random, which violates the independence assumption, and justifies moving to spatial regression.

In [ ]:
# Map OLS residuals 
# Extract residuals from the OLS model object
df_clean['ols_residuals'] = ols.u 

# Create a GeoDataFrame for plotting
gdf_clean= gpd.GeoDataFrame(
    df_clean, 
    geometry=gpd.points_from_xy(df_clean.longitude, df_clean.latitude), 
    crs="EPSG:4326"
)

# Plot the residuals
fig, ax = plt.subplots(figsize=(12, 10))

# We reproject to Web Mercator (EPSG:3857) to align with Contextily basemaps
gdf_web = gdf_clean.to_crs(epsg=3857)

gdf_web.plot(
    column='ols_residuals', 
    scheme='std_mean', # Standard deviation from mean highlights outliers well
    k=5, 
    cmap='coolwarm',   # Red = High/Positive, Blue = Low/Negative
    legend=True, 
    ax=ax, 
    markersize=15, 
    alpha=0.8
)

# Add a basemap for context
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_title('Spatial Distribution of OLS Residuals\n(Clusters indicate Spatial Autocorrelation)', fontsize=15)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the OLS residuals
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
# Plot the residuals
#gdf_clean.plot(column='ols_resid', 
 #             scheme='quantiles',
     #         k=5,
      #        markersize=8,
        #      legend=True,
         #     ax=ax,
          #    alpha=0.7)

gdf_web= gdf_clean.to_crs(epsg=3857) 


gdf_web.plot(
    column='ols_residuals', 
    scheme='quantiles', # Standard deviation from mean highlights outliers well
    k=5, 
    cmap='coolwarm',   # Red = High/Positive, Blue = Low/Negative
    legend=True, 
    ax=ax, 
    markersize=15, 
    alpha=0.8
)

# Add a basemap for context
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_title('OLS Residuals — Point-Level Listings', fontsize=16)
ax.set_axis_off()
plt.show()


In [ ]:
# Moran's I test for spatial autocorrelation in OLS residuals
residuals = ols.u.flatten()
moran = Moran(residuals, w_knn)

print("=== Global Moran's I Test on OLS Residuals ===")
print(f"Moran's I:     {moran.I:.4f}")
print(f"Expected I:    {moran.EI:.4f}")
print(f"Z-score:       {moran.z_sim:.4f}")
print(f"P-value:       {moran.p_sim:.4f}")
print()

Moran's I is 0.287 ($p = 0.001$) and residuals are significantly spatially clustered, indicating that OLS standard errors are unreliable and warns for spatial regression.

### 8. Spatial Regression Models

The Lagrange Multiplier (LM) tests from the OLS output guide model selection following the Anselin–Florax decision rule. All three models (SEM, SAR, SDM) are estimated for comparison. The interpretation of all results is presented in Section 8.6. 

#### 8.1 Spatial Error Model (SEM)

In [ ]:


# Run SEM
sem = ML_Error(
    y, x,
    w=w_knn,
    name_y=y_col,
    name_x=x_cols,
    name_ds='Edinburgh Airbnb'
)

print(sem.summary)

#### 8.2 Spatial Lag Model (SAR)

In [ ]:
# RUN SAR
sar = ML_Lag(
    y, x,
    w=w_knn,
    name_y=y_col,
    name_x=x_cols,
    name_ds='Edinburgh Airbnb'
)

print(sar.summary)


#### 8.3 Spatial Durbin Model (SDM)

In [ ]:

# spreg doesn't have a direct SDM class, so we manually create WX 
# and include it in the ML_Lag model

wx_cols = ['W_' + col for col in x_cols]
wx_array = np.column_stack([lag_spatial(w_knn, x[:, i]) for i in range(x.shape[1])])

# Combine X and WX
x_sdm = np.hstack([x, wx_array])
x_sdm_names = x_cols + wx_cols


In [ ]:
# Estimate SDM as ML_Lag with both X and WX
sdm = ML_Lag(
    y, x_sdm,
    w=w_knn,
    name_y=y_col,
    name_x=x_sdm_names,
    name_ds='Edinburgh Airbnb (SDM)'
)

print(sdm.summary)

#### 8.4 Likelihood Ratio Tests

In [ ]:
# ---- Likelihood Ratio Tests ----
print("=" * 60)
print("LIKELIHOOD RATIO (LR) TESTS")
print("=" * 60)

# LR Test 1: SDM vs SAR (H0: theta = 0, WX terms are unnecessary)
lr_sdm_sar = -2 * (sar.logll - sdm.logll)
df_diff_sar = len(x_cols)  # number of WX variables added
p_sdm_sar = 1 - stats.chi2.cdf(lr_sdm_sar, df_diff_sar)

print(f"\nLR Test: SDM vs SAR")
print(f"  H0: θ = 0 (spatially lagged X variables are unnecessary)")
print(f"  LR statistic: {lr_sdm_sar:.4f}")
print(f"  Degrees of freedom: {df_diff_sar}")
print(f"  P-value: {p_sdm_sar:.4f}")
if p_sdm_sar < 0.05:
    print("  → Reject H0: SDM is preferred over SAR")
else:
    print("  → Cannot reject H0: SAR is sufficient (SDM not needed)")

# LR Test 2: SDM vs SEM
lr_sdm_sem = -2 * (sem.logll - sdm.logll)
df_diff_sem = len(x_cols) + 1  # WX terms + rho
p_sdm_sem = 1 - stats.chi2.cdf(lr_sdm_sem, df_diff_sem)

print(f"\nLR Test: SDM vs SEM")
print(f"  H0: θ + ρβ = 0 (common factor hypothesis)")
print(f"  LR statistic: {lr_sdm_sem:.4f}")
print(f"  Degrees of freedom: {df_diff_sem}")
print(f"  P-value: {p_sdm_sem:.4f}")
if p_sdm_sem < 0.05:
    print("  → Reject H0: SDM is preferred over SEM")
else:
    print("  → Cannot reject H0: SEM is sufficient (SDM not needed)")

# Final recommendation
print("\n" + "=" * 60)
print("FINAL MODEL RECOMMENDATION")
print("=" * 60)
if p_sdm_sar < 0.05 and p_sdm_sem < 0.05:
    print("→ SDM is preferred: Both SAR and SEM are rejected in favour of SDM.")
elif p_sdm_sar >= 0.05 and p_sdm_sem < 0.05:
    print("→ SAR is preferred: SDM does not significantly improve over SAR.")
elif p_sdm_sar < 0.05 and p_sdm_sem >= 0.05:
    print("→ SEM is preferred: SDM does not significantly improve over SEM.")
else:
    print("→ Either SAR or SEM may be sufficient. Choose based on AICccc and theory.")


Both LR tests reject the restricted models ($p < 0.001$). SDM is preferred over both SAR and SEM, confirming that spatial dependence operates through both price spillovers and neighbour characteristics.

#### 8.5 Residual Maps and Moran's I For SAR, SEM, and SMD to Compute Remaining Spatial Structure.

In [ ]:
# Getting the residual from the three models and add them to the GeoDataFrame for potential spatial plotting later
gdf_web['sar_resid'] = sar.u
gdf_web['sem_resid'] = sem.u
gdf_web['sdm_resid'] = sdm.u

In [ ]:
# Residual map for SAR model
target_column = 'sar_resid'

fig, ax = plt.subplots(figsize=(12, 10))

# Plot the residuals
gdf_web.plot(
    column=target_column, 
    cmap='coolwarm',       # Red/Blue shows over/under-predictions
    scheme='quantiles',    # Classification method
    k=5,                   # Number of classes
    markersize=5, 
    legend=True, 
    ax=ax, 
    alpha=0.7
)

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_title(f'Residual Map of SAR / Spatial Lag Model', fontsize=15)
plt.show()

In [ ]:
# Moran's I test for spatial autocorrelation in SAR residuals
residuals = sar.u.flatten()
moran = Moran(residuals, w_knn)

print("=== Global Moran's I Test on SAR Residuals ===")
print(f"Moran's I:     {moran.I:.4f}")
print(f"Expected I:    {moran.EI:.4f}")
print(f"Z-score:       {moran.z_sim:.4f}")
print(f"P-value:       {moran.p_sim:.4f}")
print()

In [ ]:
# Residual map for SEM model
target_column = 'sem_resid'

fig, ax = plt.subplots(figsize=(12, 10))

# Plot the residuals
gdf_web.plot(
    column=target_column, 
    cmap='coolwarm',       # Red/Blue shows over/under-predictions
    scheme='quantiles',    # Classification method
    k=5,                   # Number of classes
    markersize=5, 
    legend=True, 
    ax=ax, 
    alpha=0.7
)

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_title(f'Residual Map of Spatial Error Model', fontsize=15)
plt.show()

In [ ]:
# Residual map for SEM model
target_column = 'sem_resid'

fig, ax = plt.subplots(figsize=(12, 10))

# Plot the residuals
gdf_web.plot(
    column=target_column, 
    cmap='coolwarm',       # Red/Blue shows over/under-predictions
    scheme='quantiles',    # Classification method
    k=5,                   # Number of classes
    markersize=5, 
    legend=True, 
    ax=ax, 
    alpha=0.7
)

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_title(f'Residual Map: {target_column}', fontsize=15)
plt.show()

In [ ]:
# Moran's I test for spatial autocorrelation in SEM residuals
residuals = sem.u.flatten()
moran = Moran(residuals, w_knn)

print("=== Global Moran's I Test on SEM Residuals ===")
print(f"Moran's I:     {moran.I:.4f}")
print(f"Expected I:    {moran.EI:.4f}")
print(f"Z-score:       {moran.z_sim:.4f}")
print(f"P-value:       {moran.p_sim:.4f}")
print()

In [ ]:
# Residual map for SDM model
target_column = 'sdm_resid'

fig, ax = plt.subplots(figsize=(12, 10))

# Plot the residuals
gdf_web.plot(
    column=target_column, 
    cmap='coolwarm',       # Red/Blue shows over/under-predictions
    scheme='quantiles',    # Classification method
    k=5,                   # Number of classes
    markersize=5, 
    legend=True, 
    ax=ax, 
    alpha=0.7
)

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_title(f'Residual Map: {target_column}', fontsize=15)
plt.show()

In [ ]:
# Moran's I test for spatial autocorrelation in SDM residuals
residuals = sdm.u.flatten()
moran = Moran(residuals, w_knn)

print("=== Global Moran's I Test on SDM Residuals ===")
print(f"Moran's I:     {moran.I:.4f}")
print(f"Expected I:    {moran.EI:.4f}")
print(f"Z-score:       {moran.z_sim:.4f}")
print(f"P-value:       {moran.p_sim:.4f}")
print()

#### 8.6 Results Interpretation: OLS vrs SAR vrs SEM vrs SMD 

OLS

The residual map shows strong spatial clustering of under‑ and over‑predicted prices across Edinburgh, rather than random noise. Moran’s I for OLS residuals is 0.287 (p = 0.001, z = 33.34), also providing clear statistical evidence of positive spatial autocorrelation and allowing rejection of spatial randomness. This implies OLS is misspecified; th residual dependence violates the independence assumption and points to a missing spatial dimension, either through price spillovers or omitted, spatially correlated factors.

SEM

SEM addresses this by modelling spatial correlation in the error term, interpreted as unobserved neighbourhood attributes shared by nearby listings; the parameter  captures this residual dependence and, unlike a spatial lag, does not imply feedback through prices themselves. After accounting for spatial error dependence, the SEM attains a pseudo of about 0.56 (close to the OLS 0.57) and a higher log‑likelihood (−1820.05 vs −2181.71), indicating a better‑fitting specification for Edinburgh Airbnb prices.
The SEM λ of about 0.508 ($p < 0.001$) shows strong positive spatial autocorrelation in the errors, meaning roughly half of each listing’s unexplained variation is shared with nearby listings due to omitted neighbourhood factors such as amenities, transport, and tourism infrastructure. The very high z‑score and tiny p‑value confirm that modelling spatial error dependence is necessary. Key predictors (`bathrooms`, `beds`, room-type dummies) remain significant with stable coefficients. 
Key predictors (`bathrooms`, `beds`, room-type dummies) remain significant with stable coefficients.

SAR

In contrast, the SAR model assumes prices themselves are mutually dependent across space, capturing competitive “copying” behaviour as hosts adjust their prices in response to nearby listings, which improves fit: the SAR attains a higher pseudo R² (0.64 overall, 0.56 for the spatial component), lower error variance and standard error, and better likelihood and information criteria than OLS, though SEM still achieves the lowest AICc/BIC and is therefore preferred once model complexity is taken into account. $\rho is 0.361$ ($p < 0.001$), indicating significant positive spatial dependence. SAR fits better than OLS but Moran's I of residuals (0.07) still indicates some remaining structure. 

SDM 

The Spatial Durbin Model (SDM) is the most general of the spatial regressions estimated, capturing both price spillovers between competing listings and neighbourhood effects arising from nearby properties. Allowing variable‑specific spatial spillovers substantially improves model fit over OLS, SEM, and SAR, with many price spillovers and neighbour‑effects proving important. The SDM explains about 66% of the variance in log‑price (pseudo), with a spatial pseudo of 0.60, and attains higher log‑likelihood and lower AICc/Schwarz than SAR and SEM, making it the statistically strongest specification.  $\rho is 0.447$ ($p < 0.001$), confirming that neighbours' attributes independently affect pricing. 



LR test

Likelihood Ratio tests show that SDM is statistically preferred to both SAR and SEM. Comparing SDM to SAR, the null that neighbours’ contextual effects are unnecessary is rejected (LR = 252.09, df = 14, p < 0.0001), so SAR is too restrictive. Comparing SDM to SEM, the common-factor restriction is also rejected (LR = 144.10, df = 15, p < 0.0001), ruling out a purely error‑driven spatial process and again favouring SDM. Together, these results justify keeping the full SDM, where price‑specific spatial spillovers are explicitly modelled rather than summarised by a single ρ or λ.
Residual diagnostics complement the LR tests. A well‑specified spatial model should yield residuals that are approximately spatially random, so residual maps and Global Moran’s I are examined for SAR, SEM, and SDM. SAR residuals have Moran’s I ≈ 0.07 (p = 0.001), indicating that the spatial lag structure absorbs much of the dependence but some clustering remains, likely due to omitted spatially correlated covariates. SEM residuals show a higher Moran’s I ≈ 0.31 (p = 0.001), meaning substantial spatial clustering persists even after modelling spatial error dependence, and its residual performance is worse than SAR’s. This pattern implies that the dominant spatial process in Edinburgh Airbnb prices is direct price interaction between nearby listings (a lag mechanism), rather than dependence confined to the error term, reinforcing the case for SDM’s richer spillover structure.



### 9. SDM Coefficient Interpretation

The SDM is the preferred global model. Comparing coefficients across OLS, SAR, SEM, and SDM reveals how spatial dependence alters estimates. However, all global models assume spatially stationary coefficients, while GWR and MGWR relax this assumption. 

There is no significant direct effect of Superhost status on price once you control for other listing traits and spatial components  (coefficient=−0.0036,p≈0.77).Each additional bathroom is associated with roughly  higher price by about (1314times=100x(exo(0.271)-1)),(coefficient=0.1271,p < 0.001). Instant Book listings are about  more expensive (56 times=100x(exo(0.0555)-1)) holding everything else (including spatial effects) constant  (coefficient=0.0555,p < 0.001) 
Each extra bed increases price by about 10 timestimes=100x(exo(0.271)-1) (coefficient=0.0941,<0.001). M
Locally, these amenities and booking convenience matter, but Superhost status does not.


In [ ]:
def get_coefs(model, names):
    coefs = {}
    for i, name in enumerate(names):
        beta = model.betas[i][0] if hasattr(model.betas[i], '__len__') else model.betas[i]
        try:
            # Try z_stat first (spatial models), then t_stat (OLS)
            if hasattr(model, 'z_stat'):
                p = model.z_stat[i][1]
            else:
                p = model.t_stat[i][1]
            sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
        except:
            sig = ''
        coefs[name] = f"{beta:.4f}{sig}"
    return coefs

In [ ]:
var_names = ['CONSTANT'] + x_cols
sdm_var_names = ['CONSTANT'] + x_sdm_names

ols_coefs = get_coefs(ols, var_names)
sar_coefs = get_coefs(sar, var_names)
sem_coefs = get_coefs(sem, var_names)
sdm_coefs = get_coefs(sdm, sdm_var_names)

all_vars = sdm_var_names
comparison = pd.DataFrame({
    'OLS': {v: ols_coefs.get(v, '—') for v in all_vars},
    'SAR': {v: sar_coefs.get(v, '—') for v in all_vars},
    'SEM': {v: sem_coefs.get(v, '—') for v in all_vars},
    'SDM': {v: sdm_coefs.get(v, '—') for v in all_vars}
})

# Add spatial parameter

comparison.loc['ρ (rho) - Spatial Lag', :] = [
    '—', 
    f"{sar.rho:.4f}", 
    '—', 
    f"{sdm.rho:.4f}"
]

comparison.loc['λ (lambda) - Spatial Error', :] = [
    '—', 
    '—', 
    f"{sem.lam:.4f}", 
    '—'
]

print("Coefficient Comparison: OLS vs SAR vs SEM vs SDM")
print("(*** p<0.01, ** p<0.05, * p<0.1)")
print("=" * 80)
print(comparison.to_string())

# Part 2: Point-Level GWR and MGWR


GWR and MGWR allow regression coefficients to vary across space, capturing spatial heterogeneity that global models cannot. Here, each listing (point) is the unit of analysis.

In [ ]:
#pip install mgwr

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import libpysal as ps 
from libpysal.weights import Queen
from esda.moran import Moran
import statsmodels.api as sm

# MGWR functions
from mgwr.gwr import GWR,MGWR
from mgwr.sel_bw import Sel_BW

#### 1. Data Preparation

Each Airbnb listing is used directly as a point observation in projected coordinates (EPSG:27700),  for kernel distance computation.

In [ ]:
df_clean.shape

In [ ]:
# 2. Convert listings (points) to GeoDataFrame (points)
gdf_clean = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.points_from_xy(df_clean['longitude'], df_clean['latitude']),
    crs='EPSG:4326'
).to_crs(epsg=27700)

In [ ]:
gdf_clean.crs

#### 2. Variable Selection

The same predictor set from Part 1 is used, ensuring comparability across global and local frameworks.

In [ ]:
x_cols = ['is_superhost', 
'is_instant_bookable',
'bathrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d',  
'calculated_host_listings_count', 'reviews_per_month',     
] + room_dummy_cols
variable_names = x_cols

y = gdf_clean [['log_price']].values

X = gdf_clean [variable_names].values



#### 3. Zero-Variance Check

In [ ]:
# Before standardising, I ensure X and y are clean numpy float64 arrays, so I remove zero-variance columns from variable list if any exist, 
# as they are likely to be pandas DataFrames or contain mixed types. 
X =df_clean[variable_names].values.astype(np.float64)
y = df_clean[['log_price']].values.astype(np.float64)


#### 4. Standardisation

Both $X$ and $y$ are standardised to produce unit-free, comparable coefficients across variables and locations.

In [ ]:
# Standardise X and y
X = (X - X.mean(axis=0))/X.std(axis=0)
y = (y - y.mean(axis=0))/y.std(axis=0)


X = (X - X.mean(axis=0)) / X.std(axis=0)
y = (y - y.mean(axis=0)) / y.std(axis=0)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"X mean (should be ~0): {X.mean(axis=0).round(6)}")
print(f"y mean (should be ~0): {y.mean().round(6)}")

In [ ]:
gdf_clean.shape

#### 6. Coordinates

In [ ]:
# Extract projected coordinates directly from point geometries for kernel distance computation.
# .x = Easting (metres), .y = Northing (metres)
coords = np.column_stack([
    gdf_clean.geometry.x.values,
    gdf_clean.geometry.y.values
])

gdf_clean['proj_X'] = gdf_clean.geometry.x.values
gdf_clean['proj_Y'] = gdf_clean.geometry.y.values

print(f"Coordinates shape: {coords.shape}")
print(f"X range: {coords[:,0].min():.0f} – {coords[:,0].max():.0f} m")
print(f"Y range: {coords[:,1].min():.0f} – {coords[:,1].max():.0f} m")
#print(gdf_clean['proj_X'])

#### 7. OLS Baseline (Point Level - See Part 1)

In [ ]:
gdf_clean.crs

In [ ]:
gdf_web.crs

In [ ]:
gdf_web.shape

In [ ]:
df_clean.shape


#### 8. GWR Model

Bandwidth is selected via AICc minimisation using an adaptive bisquare kernel which identifies number of nearest neighbours.

In [ ]:
%%time

# Step 1: Select optimal bandwidth
gwr_selector = Sel_BW(coords, y, X)
gwr_bw = gwr_selector.search(verbose=True, criterion='AICc')

print(f"\nSelected optimal GWR bandwidth: {gwr_bw} nearest neighbours")

The optimal bandwidth is 89 nearest neighbours.

In [ ]:
# Fit the GWR model with the selected bandwidth
gwr_results = GWR(coords, y, X, bw=gwr_bw,name_x=variable_names).fit()

#### 9. GWR Summary

GWR reports both global OLS and local results. Key diagnostics are adjusted $R^2$, AICc, and local coefficient distributions.

In [ ]:
gwr_results.summary()

In [ ]:
variable_names

In [ ]:
gwr_results.params[:,4] #this will give you location-specific coefficient for pct_bach

In [ ]:
from matplotlib import colors

ax = gdf_clean.plot(column=gwr_results.params[:,4],figsize=(10,5),legend=True, 
                     linewidth=0.0,aspect=1)

plt.title("Coefficients of " + variable_names[3] ,fontsize=12)

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mgwr.utils import shift_colormap,truncate_colormap
from matplotlib import cm,colors

def param_plots(result, gdf, names=[], filter_t=False, figsize=(10, 10)):
    
    #Size of the plot. Here we have a 2 by 2 layout.
    k = gwr_results.k
    
    fig, axs = plt.subplots(int(k/2)+1, 2, figsize=figsize)
    axs = axs.ravel()
    
    #The max and min values of the color bar.
    vmin = -0.8
    vmax = 0.8
    
    cmap = cm.get_cmap("bwr_r")
    norm = colors.BoundaryNorm(np.arange(-0.8,0.9,0.1),ncolors=256)
    
    for j in range(k):
        
        pd.concat([gdf,pd.DataFrame(np.hstack([result.params,result.bse]))],axis=1).plot(ax=axs[j],column=j,vmin=vmin,vmax=vmax,
                                                                                         cmap=cmap,norm=norm,linewidth=0.1,edgecolor='white',aspect=1)
        axs[j].set_title("Parameter estimates of \n" + names[j],fontsize=10)
        
        if filter_t:
            rslt_filtered_t = result.filter_tvals()
            if (rslt_filtered_t[:,j] == 0).any():
                gdf[rslt_filtered_t[:,j] == 0].plot(color='lightgrey', ax=axs[j],linewidth=0.1,edgecolor='white',aspect=1)
        
        plt.axis('off')
    
    fig = axs[j].get_figure()
    cax = fig.add_axes([0.99, 0.2, 0.025, 0.6])
    sm = plt.cm.ScalarMappable(cmap=cmap,norm=norm)
    # fake up the array of the scalar mappable. Urgh...
    sm._A = []
    fig.colorbar(sm, cax=cax)

#### 10. GWR Coefficient Maps


In [ ]:

# Positive (negative) effects in red (blue); intensity reflects magnitude. Spatial variation indicates non-stationarity.
param_plots(gwr_results, gdf_clean, names=['intercept'] + variable_names,figsize = (10,40))

#### 11. GWR Residuals

In [ ]:
#Here we use the K-nearest neigbourus because it refers to points 
w = KNN.from_array(coords, k=8)

#row standardization
w.transform = 'R'

residual_moran = Moran(gwr_results.resid_response.reshape(-1), w)
residual_moran.I

#### 12. GWR Inference

Significance-filtered maps ($p < 0.05$, corrected for multiple testing). Grey areas indicate insignificant local coefficients.

In [ ]:
param_plots(gwr_results, gdf_clean, names=['intercept'] + variable_names, figsize = (10,40), filter_t=True)

#### 13. MGWR Model

MGWR allows each covariate its own bandwidth, capturing multiscale spatial processes. Computationally intensive for large datasets.

In [ ]:
%%time
mgwr_selector = Sel_BW(coords, y, X, multi=True)
mgwr_bw = mgwr_selector.search(verbose=True)

print("Selected optimal bandwidth is:", mgwr_bw)

In [ ]:
%%time

mgwr_results = MGWR(coords, y, X, selector=mgwr_selector,name_x=variable_names).fit()

#### MGWR Summary

Each covariate has a distinct bandwidth: large bandwidths indicate near-global effects; small bandwidths indicate highly localised relationships. Adjusted $t$-values correct for multiple testing.

In [ ]:
#Here we use the K-nearest neigbourus because it refers to points 
w = KNN.from_array(coords, k=8)

#row standardization
w.transform = 'R'

residual_moran = Moran(mgwr_results.resid_response.reshape(-1), w)
residual_moran.I

MGWR residual Moran's I approaches zero — the multiscale specification fully captures the spatial structure.

In [ ]:
param_plots(mgwr_results,gdf_clean, names=['intercept'] + variable_names,figsize = (10,15), filter_t=True)

### OLS vs. GWR vs. MGWR (Point Level)

MGWR consistently outperforms GWR and OLS: higher adjusted $R^2$, lower AICc, and near-zero residual Moran's I.

# Part 2: Polygone-Level (Data Zone) GWR and MGWR


For the DataZone-level analysis below, listings are spatially joined to 2022 DataZone polygons, aggregated to zone-level medians, and merged back onto polygon geometries.

## 1 DataZone Aggregation Pipeline and Rationale

In [ ]:
#Individual Airbnb listings (point) are spatially joined to Scotland's 2022 DataZone boundaries (polygon) using a point-in-polygon operation. 
# The joined data is then aggregated to the DataZone level using median values for continuous variables and modal values for categorical 
# variables (room type). The zone-level statistics which are the required spatial unit for GWR/MGWR calibration, are merged back onto the 
# polygon geometries to preserve the spatial information needed for mapping, spatial weights construction, and centroid extraction. Aggregation
#  avoids spatial pseudoreplication (multiple non-independent listings per zone) and enables Queen contiguity weights for polygon-based Moran's I.

Belwo I follow exactly the same procedure as above.
# 1. Load the DataZone shapefile — this becomes gdf_zones
shp = gpd.read_file('/Users/xrysa/Desktop/MSc_in_Urban_Analytics/Advanced_Topics/2_3_week_excersise/SG_DataZoneBdry_2022/SG_DataZone_Bdry_2022.shp')

#if column is called something else, search for it ───────────────
# Run this to find which column contains Edinburgh
for col in shp.columns:
    if shp[col].astype(str).str.contains('Edinburgh', case=False).any():
        print(f"Found 'Edinburgh' in column: '{col}'")

shp = shp.reset_index(drop=True)
print(f"Edinburgh DataZones: {len(shp)}")   # ✅ should be ~940 zones


shp  = shp.to_crs(epsg=27700)
print(shp.crs)    # EPSG:27700
print(shp.shape)  # (n_zones, n_cols)

# 2. Convert listings to GeoDataFrame
gdf_clean = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.points_from_xy(df_clean['longitude'], df_clean['latitude']),
    crs='EPSG:4326'
).to_crs(epsg=27700)

# 3. Spatial join listings → DataZones
# Result: Every point in  gdf_clean  now has columns from  shp  appended—including a  dzcode  (Data Zone code)
#  and an  index_right  column showing which polygon index it matched
# how='left' : Keeps all points from  gdf_clean ; fills with  NaN  for points that don’t fall inside any polygon
# predicate='within' : Matches only points completely inside polygons (returns  True  when a point geometry is contained 
# within a polygonMatches only points completely inside polygons (returns  True  when a point geometry is contained within a polygon
gdf_joined_spj= gpd.sjoin(gdf_clean, shp , how='left', predicate='within')
#gdf_joined_spj = gdf_joined_spj.dropna(subset=['dzcode']).drop(columns='index_right') 

# #  Because OLS and Moran’s I operate on zones (polygons), not individual listings (points), I need to
# 4. Aggregate to zone level. before gdf_zones has unit datazone (polygone) while df_clea or shp have unit  a listing (point)

df_zone_stats = (gdf_joined_spj
                 .groupby('dzcode')
                 .agg({
                     # ── Airbnb metrics ──────────────────────
                     'price':                'median',
                     'number_of_reviews':    'median',
                     'availability_365':     'median',
                     'minimum_nights':       'median',
                     'reviews_per_month':    'median',
                     'host_listings_count':  'median',
                     'host_total_listings_count':  'median',
                     'bathrooms':             'median', 
                     'beds':                  'median',
                     'maximum_nights':        'median',
                     'estimated_revenue_l365d':'median',  
                     'calculated_host_listings_count':    'median', 
                     'is_superhost':         'median',


                     # ── Count of listings per zone ──────────
                     'id':                   'count',
                     'is_instant_bookable':   'count',
                     # ── Any other column you need ────────────
                     'room_type':            lambda x: x.mode()[0],  # most common category
                  })
                 .rename(columns={'id': 'n_listings'})
                 .reset_index())


print(shp.columns.tolist())
print(df_zone_stats.columns.tolist())


# 5. Merge back to zones → this becomes df_joined used in ALL later cells
# Merge onto shp (polygons) — NOT gdf_joined_spj
df_joined = shp.merge(df_zone_stats, on='dzcode', how='right')
print(df_joined.crs)   # EPSG:27700 — carries geometry from shp
print(df_joined.shape)  # 1 row per DataZone 

# 6. Because  room_dummy_cols  were created at the listing level (in  df_clean  or  gdf_joined_spj ) 
# but never carried through into  df_joined . The aggregation step only kept  room_type  as a mode string, not as dummies.
room_dummies = pd.get_dummies(df_joined['room_type'], prefix='room', drop_first=True)
df_joined    = pd.concat([df_joined, room_dummies], axis=1)
#print(df_joined.head(3))

# Define dummy col names dynamically
room_dummy_cols = room_dummies.columns.tolist()
print(room_dummy_cols)   # ['room_Hotel room', 'room_Private room', 'room_Shared room']




In [ ]:
df_joined.crs


In [ ]:
len(df_joined)


In [ ]:
df_joined.describe()

In [ ]:
#dz_join['Name'].value_counts()

In [ ]:
# Select the dependent variable and independent variables for GWR/MGWR

variable_names =['is_superhost', 'is_instant_bookable',
'host_listings_count', 
'host_total_listings_count', 
'bathrooms', 'beds',
'minimum_nights', 'maximum_nights',
'availability_365','number_of_reviews', 'estimated_revenue_l365d',  
'calculated_host_listings_count', 'reviews_per_month',     
] + room_dummy_cols


y = df_joined[['price']].values

X = df_joined[variable_names].values

In [ ]:
# Detect if there is still huge issues with colinearity, which actually ids the reason I removed 'is_instant_bookable' from the list above
# import pandas as pd
import numpy as np

X_df_check = pd.DataFrame(X, columns=variable_names)

# ── Check 1: constant columns (zero variance) ─────────────────────────────────
zero_var = X_df_check.std() == 0
print("Zero variance columns:")
print(zero_var[zero_var].index.tolist())

# ── Check 2: perfectly correlated pairs ───────────────────────────────────────
corr = X_df_check.corr().abs()
high_corr = [(c1, c2) for c1 in corr.columns 
                       for c2 in corr.columns 
                       if c1 < c2 and corr.loc[c1, c2] > 0.95]
print("\nPerfectly/near-perfectly correlated pairs (>0.95):")
for pair in high_corr:
    print(f"  {pair[0]}  ↔  {pair[1]}  →  {corr.loc[pair[0], pair[1]]:.4f}")

# ── Check 3: NaN or inf ───────────────────────────────────────────────────────
#print(f"\nNaN in X: {np.isnan(X).sum()}")
#print(f"Inf in X: {np.isinf(X).sum()}")

# ── Check 4: rank deficiency ──────────────────────────────────────────────────
##print(f"\nMatrix rank: {np.linalg.matrix_rank(X)}")
#print(f"Num columns: {X.shape[1]}")
# rank must equal num columns — if less, matrix is singular


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Ensure you have a constant (intercept) for the VIF calculation
# VIF is usually calculated with an intercept to get accurate results
X_df = gdf_joined_spj[variable_names].copy()
# it does not work for df_joined, because X_df  contains non-numeric, NaN, or boolean columns that numpy cannot process. 
X_df['intercept'] = 1

# Calculate VIF for each variable
vif_data = pd.DataFrame()
vif_data["feature"] = X_df.columns
vif_data["VIF"] = [variance_inflation_factor(X_df.values, i) 
                          for i in range(len(X_df.columns))]

# View the results (excluding the intercept row for clarity)
print(vif_data[vif_data['feature'] != 'intercept'])

In [ ]:
# Ensure X and y are clean numpy float64 arrays 
# not plain numpy float arrays — they are likely pandas DataFrames or contain mixed types. 
# The fix is to explicitly cast to  float64  before standardising
X =df_joined [variable_names].values.astype(np.float64)
y = df_joined[['price']].values.astype(np.float64)

X = (X - X.mean(axis=0))/X.std(axis=0)
y = (y - y.mean(axis=0))/y.std(axis=0)

In [ ]:
# Extract projected X, Y coordinates from geometry 
# geometry is already in EPSG:27700 (British National Grid — metres)
# .x = Easting, .y = Northing
# this part of the code DOES NOT work for df_jointed, because  .x  and  .y  only work on Point geometries, 
# but  df_joined  has Polygon geometries (DataZone polygons). I need to extract the centroid of each polygon first:
#df_joined['proj_X'] = df_joined['geometry'].x   # Easting  (metres)
#df_joined['proj_Y'] = df_joined['geometry'].y   # Northing (metres)

# .centroid converts each polygon 
df_joined['proj_X'] = df_joined['geometry'].centroid.x   # Easting  (metres)
df_joined['proj_Y'] = df_joined['geometry'].centroid.y   # Northing (metres)

coords = df_joined [['proj_X', 'proj_Y']].values

In [ ]:
df_joined.crs

Centroids provide the single coordinate pair per zone required for GWR kernel weighting.

#### OLS and Residual Diagnostics (DataZone Level)

In [ ]:
# Fit the Global OLS model
# We add a constant (intercept) to the independent variables
X_ols = sm.add_constant(X)
ols_model = sm.OLS(y, X_ols).fit()

# Extract residuals and assign them to the GeoDataFrame
df_joined ['ols_resid'] = ols_model.resid

# Visualize the OLS residuals
fig, ax = plt.subplots(1, 1, figsize=(12, 8))


# Plot the residuals
df_joined.plot(
    column='ols_resid', 
    cmap='coolwarm',       # Red/Blue shows over/under-predictions
    k=5,                   # Number of classes
    markersize=5, 
    legend=True, 
    ax=ax, 
    alpha=0.7
)

state = df_joined.dissolve(by='dzcode').geometry.boundary
# Add basemap
state.plot(ax=ax, edgecolor='black', facecolor='none', linewidth=0.5)

ax.set_title('OLS Regression Residuals — Edinburgh Airbnb', fontsize=16)
ax.set_axis_off()
plt.show()

# Calculate Global Moran's I for the OLS residuals
# Create a Queen contiguity spatial weights matrix from the county geometries
w = Queen.from_dataframe(df_joined)
w.transform = 'r'  # Row-standardize the weights

# Calculate the Moran's I statistic
moran_ols = Moran(df_joined['ols_resid'], w)

print(f"Global Moran's I for OLS residuals: {moran_ols.I:.4f}")
print(f"Expected value (E[I]): {moran_ols.EI:.4f}")
print(f"p-value (pseudo-significance): {moran_ols.p_sim:.5f}")


#### GWR — DataZone Level

In [ ]:
%%time

gwr_selector = Sel_BW(coords, y, X,)

gwr_bw = gwr_selector.search(verbose=True,criterion='AICc')

print("Selected optimal bandwidth is:", gwr_bw)

Optimal bandwidth: 178 nearest DataZones.

In [ ]:
# Fit the GWR model with the selected bandwidth
gwr_results = GWR(coords, y, X, bw=gwr_bw,name_x=variable_names).fit()

In [ ]:
gwr_results.summary()

In [ ]:
variable_names

In [ ]:
gwr_results.params[:,4] #this will give you location-specific coefficient for pct_bach

In [ ]:
from matplotlib import colors

ax = df_joined.plot(column=gwr_results.params[:,4],figsize=(10,5),legend=True, 
                     linewidth=0.0,aspect=1)

plt.title("Coefficients of " + variable_names[3] ,fontsize=12)

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mgwr.utils import shift_colormap,truncate_colormap
from matplotlib import cm,colors

def param_plots(result, gdf, names=[], filter_t=False, figsize=(10, 10)):
    
    #Size of the plot. Here we have a 2 by 2 layout.
    k = gwr_results.k
    
    fig, axs = plt.subplots(int(k/2+1), 2, figsize=figsize) #orginally had (k/2)+1 procuding one extra empty graph
    axs = axs.ravel()
    
    #The max and min values of the color bar.
    vmin = -0.8
    vmax = 0.8
    
    cmap = cm.get_cmap("bwr_r")
    norm = colors.BoundaryNorm(np.arange(-0.8,0.9,0.1),ncolors=256)
    
    for j in range(k):
        
        pd.concat([gdf,pd.DataFrame(np.hstack([result.params,result.bse]))],axis=1).plot(ax=axs[j],column=j,vmin=vmin,vmax=vmax,
                                                                                         cmap=cmap,norm=norm,linewidth=0.1,edgecolor='white',aspect=1)
        axs[j].set_title("Parameter estimates of \n" + names[j],fontsize=10)
        
        if filter_t:
            rslt_filtered_t = result.filter_tvals()
            if (rslt_filtered_t[:,j] == 0).any():
                gdf[rslt_filtered_t[:,j] == 0].plot(color='lightgrey', ax=axs[j],linewidth=0.1,edgecolor='white',aspect=1)
        
        plt.axis('off')
    
    fig = axs[j].get_figure()
    cax = fig.add_axes([0.99, 0.2, 0.025, 0.6])
    sm = plt.cm.ScalarMappable(cmap=cmap,norm=norm)
    # fake up the array of the scalar mappable. Urgh...
    sm._A = []
    fig.colorbar(sm, cax=cax)

In [ ]:
gwr_results.k

#### GWR Coefficient Maps — DataZone Level

In [ ]:
param_plots(gwr_results, df_joined, names=['intercept'] + variable_names,figsize = (10,40))
plt.subplots_adjust(hspace=0.4)

In [ ]:
#Here we use the Queen contiguity
w = Queen.from_dataframe(df_joined)

#row standardization
w.transform = 'R'

residual_moran = Moran(gwr_results.resid_response.reshape(-1), w)
residual_moran.I

#### GWR Inference — DataZone Level

In [ ]:
param_plots(gwr_results, df_joined, names=['intercept'] + variable_names, figsize = (10,40), filter_t=True)
plt.subplots_adjust(hspace=0.4)

#### MGWR — DataZone Level

In [ ]:
%%time
mgwr_selector = Sel_BW(coords, y, X, multi=True)
mgwr_bw = mgwr_selector.search(verbose=True)

print("Selected optimal bandwidth is:", mgwr_bw)

In [ ]:
%%time

mgwr_results = MGWR(coords, y, X, selector=mgwr_selector,name_x=variable_names).fit()

In [ ]:
mgwr_results.summary()

In [ ]:
#Here we use the Queen contiguity
w = Queen.from_dataframe(df_joined)

#row standardization
w.transform = 'R'

residual_moran = Moran(mgwr_results.resid_response.reshape(-1), w)
residual_moran.I

In [ ]:
param_plots(mgwr_results, df_joined, names=['intercept'] + variable_names,figsize = (10,40), filter_t=True)
plt.subplots_adjust(hspace=0.4)

### OLS vs. GWR vs. MGWR (DataZone Level)

MGWR again outperforms both OLS and GWR at the DataZone level.

In [ ]:
#References


#Luo, J., et al. (2023). A Multi-Source Information Learning Framework for Airbnb Price Prediction. Expert Systems with Applications.
#Perez-Sanchez, V. R., et al. (2018). The What, Where, and Why of Airbnb Price Determinants. Sustainability.
#Sainaghi, R. (2021). Analysis of price determinants in the case of Airbnb listings. Economic Research-Ekonomska Istraživanja.
#Voltes-Dorta, A., & Sánchez-Medina, A. (2020). Drivers of Airbnb prices according to property/room type, season and location: A regression approach. Journal of Hotel and Business Management.
#Wang, D., & Nicolau, J. L. (2017). Price determinants of sharing economy based accommodation rental: A 33 cities pricing model approach. International Journal of Hospitality Management.